# 🐟 DeepLabCut Project Workflow Guide

This notebook helps you run DeepLabCut (DLC) either by:
- Training a new model from your own videos, or
- Using an inherited pre-trained model to analyze new videos.

🎯 Everything here can be done via the **GUI** or in **code** — this notebook explains both.

If you already have a trained model (with `dlc-models/` and `config.yaml`), skip ahead to the **"Using an Inherited DLC Project"** section.

## Activating the environment

Before starting, you need to activate the environment created for DLC (refer to the [`readme.md`](./readme.md) file).

Once the environment has been created, in the terminal/command prompt run: `conda activate DEEPLABCUT`

Before running any cells in this notebook, make sure you've selected the correct environment (kernel) where DeepLabCut is installed.

In this Jupyter Notebook:
- Click the **kernel name** in the top right corner (e.g., `Python 3 (ipykernel)`)
- Choose the environment you created for DLC (e.g., `DEEPLABCUT`)

## Starting from Scratch with a new DLC project

#### Step 1: Start by importing deeplabcut

In [3]:
import deeplabcut
print(deeplabcut.__version__)

ModuleNotFoundError: No module named 'deeplabcut'

#### Step 2: Create a New Project
Creates a new folder containing all your project files.

> **GUI equivalent**:
Click “Create New Project”
Fill in the name, author, select video(s), and choose where to save it

In [1]:
from pathlib import Path

# CHANGE THESE ALL of these to match your setup:

my_project = 'MyProject'  # Name of your DLC project
my_name = 'YourName'      # Your name

# Path to the video(s) you want to analyze
video_path = [str(Path('/Users/yourname/Videos/fish_trial_01.mp4'))]  # List of videos

# Folder where you want the project to be saved
project_path = Path('/Users/yourname/Documents/DLC_Projects/FishPoseProject')

# DLC automatically places config.yaml inside the project folder it creates
path_to_config = project_path / my_project / 'config.yaml'

In [ ]:
deeplabcut.create_new_project(
    my_project,  
    my_name,   
    video_path, 
    working_directory= project_path,
    copy_videos=True #all videos will also be saved into porject folder, can change to false
)

**After this step:** update config.yaml with the list of bodyparts, number of frames to pick (`numframes2pick`), and add skeleton if needed (means?) - no need if on GUI 

#### Step 3: Extract Frames
Select frames to label.

💡 **How many frames?**  
Recommended to extract **~20–50 frames per video**, depending on:
- The duration of the video
- The variety of poses/behaviours you expect

To control this, open `config.yaml` and set:
```yaml
numframes2pick: 30  # Example: change this to your preferred number
```

**kmeans clustering** is used to extract visually different frames, helping you label a diverse set of poses for a better model eventually.



> **GUI equivalent**:
Click “Extract Frames”
Choose automatic or manual, and pick your method (e.g. kmeans)

In [ ]:
deeplabcut.extract_frames(
    path_to_config, 
    mode='automatic', 
    algo='kmeans', #k-means used as default
    userfeedback=False
)

#### Step 4: Label Frames
Open a GUI (opens as a pop-up) by running the line below and label the bodyparts on each of the frames. 

Tutorial on how to navigate the pop up here: https://www.youtube.com/watch?v=hsA9IB5r73E&list=PLjpMSEOb9vRE2e0wLHgB3MImAMMDJI_YT&index=2

> **GUI equivalent:**
Click “Label Frames”
Use the label window to click each keypoint per frame



In [ ]:
deeplabcut.label_frames(path_to_config)

#### Step 5: Check Frames
Visual check for mistakes or missing labels.

> **GUI equivalent:**
Click “Check Labels” to preview keypoints over images

Fix any labeling errors before proceeding.

In [ ]:
deeplabcut.check_labels(path_to_config)

#### Step 6: Create Training Dataset
Prepares the labeled data for training.

> **GUI equivalent:**
Click “Create Training Dataset”

This creates a training-datasets/ folder inside your project.

In [ ]:
deeplabcut.create_training_dataset(path_to_config)

#### Step 7: Train Network (GPU only)
🚨 Only run this on a machine with a GPU (e.g., Toothless) — training takes hours!

> **GUI equivalent:**
Click “Train Network”
Customize training iterations and intervals ?? 

Adjust displayiters, saveiters, and maxiters to suit your project size.

In [ ]:
deeplabcut.train_network(
    path_to_config,
    shuffle=1,
    displayiters=100,
    saveiters=1000,
    maxiters=200000
)

#### What do the parameters mean?

- **`shuffle=1`**  
  This lets DLC train on different random subsets of your data.  
  Keep this as `1` unless you're experimenting or comparing multiple trainings.

- **`displayiters=100`**  
  DLC will print the training status (loss, accuracy, etc.) every 100 iterations.  
  Set this lower if you want to monitor progress more frequently.

- **`saveiters=1000`**  
  A model snapshot (i.e., a saved state) will be saved every 1000 iterations.  
  This is useful in case training crashes or you want to resume training later.

- **`maxiters=200000`**  
  This is the total number of training iterations.  
  More iterations = potentially better accuracy, but longer training time.  
  - ~50,000–100,000 may be enough for basic tasks.  
  - Use 200,000+ if your labels are noisy or behavior is complex.


> 💡 You can stop and restart training using saved snapshots if needed. To do this, set the `init_weights` path in `config.yaml` to your latest snapshot file.

#### Step 8: Analyze Videos (Inference)
Use the trained model to predict keypoints on your video(s).

> **GUI equivalent:**
Click “Analyze Videos”
Select the videos you want to process

This step generates .h5 and .csv files in your videos/ folder.


In [4]:
deeplabcut.analyze_videos(path_to_config, [video_path/'video.mp4'], save_as_csv=True) ??

SyntaxError: invalid syntax (3946570723.py, line 1)

#### Step 9: Create Labeled Videos (Optional)

This step overlays predicted keypoints on the original video(s). It's useful for visually verifying whether the model is tracking the animal well.

> **GUI equivalent**:  
Click “Create Labeled Videos” and select your analyzed videos.

<i> If your videos are long or high-res, this may take time. You can always re-run this step later, even after post-processing. </i>


In [ ]:
deeplabcut.create_labeled_video(
    path_to_config,
    video_path,         # The same list of video(s) you analyzed
    draw_skeleton=True  # Set to False if you don’t want lines connecting points
)

#### Step 10: Evaluate Model Performance

After training, you should evaluate how well your model performs on labeled frames it hasn't seen during training.

> **GUI equivalent**:  
Click “Evaluate Network”.


### What does it do?
- DLC compares predicted vs. actual keypoints on your **test set** (a portion of labeled frames not used for training).
- It reports the **error in pixels** — lower = better.

**Useful for:**
- Deciding whether to stop training or label more frames
- Comparing multiple training runs (using different shuffles)

<i> You can also plot individual prediction errors per body part. </i>


In [5]:
deeplabcut.evaluate_network(
    path_to_config,
    plotting=True  # Set to False if you just want error numbers
)

NameError: name 'deeplabcut' is not defined

---

## Using an Inherited DLC Project

Use this section if you've been given a pre-trained DLC project folder. You don’t need to extract or label frames, or train a new model.

### Required folder contents:
- `config.yaml`
- `dlc-models/` with trained weights and `pose_cfg.yaml`
- Videos you want to analyze


### Step 1: Set config file and v

Set the config path (if not already set above):

In [2]:
#set this to the correct path - where your inhertied config file is saved
path_to_config = '/path/to/inherited/project/config.yaml'

#set this to folder where videos to analyse are
video_path = '/Users/yourname/Videos/fish_trial_02.mp4'

In [ ]:
import deeplabcut
cfg = deeplabcut.load_config(path_to_config)

### Step 2: Analyze new videos using the trained model

In [ ]:
deeplabcut.analyze_videos(cfg, video_path, save_as_csv=True)

### Step 3 (Optional): Create labeled videos to check performance

Especially useful to verify the inherited model works well on your videos.

In [7]:
deeplabcut.create_labeled_video(cfg, inherited_video_path, draw_skeleton=True)

NameError: name 'deeplabcut' is not defined

#### Step 5: Evaluate Model (Optional)

You can run model evaluation **only if the project includes labeled frames** (skip if you do not have labelled frames). 

> **GUI equivalent**:
Click “Evaluate Network”

If you received an inherited project with labeled data, this will:
- Compute prediction errors on test frames
- Help you decide if the model is good enough
- Suggest whether additional labeling is needed

In [ ]:
deeplabcut.evaluate_network(cfg, plotting=True)

---
### Whats next?

Once you have .csv and .h5 prediction files, head over to the [`Post_DLC.ipynb`](./Post_DLC.ipynb) notebook to:

* Clean low-likelihood points

* Segment trials

* Compute movement angles

* Generate metrics